In [3]:
import pandas as pd

df = pd.read_excel("C:/Users/Dell/feature-pipeline-retail/clean_breast_cancer_dataset.xlsx")

df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [4]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, TransformerMixin
import joblib

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")  # 0=malignant, 1=benign

X.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

baseline_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=5000))
])

baseline_scores = cross_val_score(baseline_pipe, X, y, cv=cv, scoring="roc_auc")
print(f"Baseline ROC-AUC: {baseline_scores.mean():.4f} ± {baseline_scores.std():.4f}")

Baseline ROC-AUC: 0.9953 ± 0.0053


In [6]:
%%writefile features.py
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class RatioFeatures(BaseEstimator, TransformerMixin):
    """Adds ratio features between related mean/worst columns (e.g. concavity/area)."""
    def __init__(self, pairs=None):
        self.pairs = pairs or [
            ("mean concavity", "mean area"),
            ("worst concavity", "worst area"),
            ("mean concave points", "mean perimeter"),
            ("worst concave points", "worst perimeter"),
        ]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.feature_names_in_) if not isinstance(X, pd.DataFrame) else X.copy()
        out = pd.DataFrame(index=X.index)
        for num, den in self.pairs:
            out[f"{num}_over_{den}"] = X[num] / (X[den] + 1e-9)
        return out.values

    def fit_transform(self, X, y=None):
        self.feature_names_in_ = X.columns if isinstance(X, pd.DataFrame) else None
        return self.fit(X, y).transform(X)


class WorstMeanGapFeatures(BaseEstimator, TransformerMixin):
    """Adds (worst - mean) gap features — captures tumor heterogeneity."""
    def __init__(self, bases=None):
        self.bases = bases or [
            "radius", "texture", "perimeter", "area", "smoothness",
            "compactness", "concavity", "concave points", "symmetry", "fractal dimension"
        ]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        out = pd.DataFrame(index=X.index)
        for b in self.bases:
            out[f"{b}_gap"] = X[f"worst {b}"] - X[f"mean {b}"]
        return out.values


class SkewLogTransform(BaseEstimator, TransformerMixin):
    """Applies log1p to reduce skew on right-skewed columns; fit stores nothing (safe, stateless)."""
    def __init__(self, columns=None):
        self.columns = columns

    def fit(self, X, y=None):
        self.columns_ = self.columns or list(X.columns)
        return self

    def transform(self, X):
        return np.log1p(X[self.columns_].clip(lower=0)).values

Overwriting features.py


In [7]:
from features import RatioFeatures, WorstMeanGapFeatures, SkewLogTransform

# quick smoke test
rf = RatioFeatures()
print(rf.fit_transform(X).shape)

(569, 4)


In [8]:
skew_cols = ["mean area", "worst area", "mean concavity", "worst concavity"]

feature_union = ColumnTransformer([
    ("scaled_raw", StandardScaler(), list(X.columns)),
    ("ratios", RatioFeatures(), list(X.columns)),
    ("gaps", WorstMeanGapFeatures(), list(X.columns)),
    ("skew", SkewLogTransform(columns=skew_cols), skew_cols),
])

enriched_pipe = Pipeline([
    ("features", feature_union),
    ("clf", LogisticRegression(max_iter=5000))
])

enriched_scores = cross_val_score(enriched_pipe, X, y, cv=cv, scoring="roc_auc")
print(f"Enriched ROC-AUC: {enriched_scores.mean():.4f} ± {enriched_scores.std():.4f}")

Enriched ROC-AUC: 0.9949 ± 0.0055


In [9]:
results = pd.DataFrame({
    "Pipeline": ["Baseline", "Enriched"],
    "Mean ROC-AUC": [baseline_scores.mean(), enriched_scores.mean()],
    "Std": [baseline_scores.std(), enriched_scores.std()],
})
results["Gain (pp)"] = (results["Mean ROC-AUC"] - baseline_scores.mean()) * 100
results

,Pipeline,Mean ROC-AUC,Std,Gain (pp)
0,Baseline,0.995314,0.005345,0.000000
1,Enriched,0.994927,0.005547,-0.038742


In [10]:
enriched_pipe.fit(X, y)
joblib.dump(enriched_pipe, "model.joblib")

['model.joblib']

In [11]:
%%writefile test_features.py
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from features import RatioFeatures, WorstMeanGapFeatures, SkewLogTransform

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)

def test_ratio_features_shape():
    rf = RatioFeatures()
    out = rf.fit_transform(X)
    assert out.shape[0] == X.shape[0]
    assert out.shape[1] == len(rf.pairs)

def test_ratio_features_no_nan():
    rf = RatioFeatures()
    out = rf.fit_transform(X)
    assert not np.isnan(out).any()

def test_gap_features_correct_values():
    wg = WorstMeanGapFeatures(bases=["radius"])
    out = wg.fit_transform(X)
    expected = (X["worst radius"] - X["mean radius"]).values
    assert np.allclose(out.flatten(), expected)

def test_skew_log_no_negative_output_error():
    sk = SkewLogTransform(columns=["mean area"])
    out = sk.fit_transform(X)
    assert out.shape[0] == X.shape[0]

def test_transformers_stateless_no_leakage():
    # fitting on a subset shouldn't change output for the same rows
    rf = RatioFeatures()
    out_full = rf.fit_transform(X)
    out_subset = rf.fit_transform(X.iloc[:100])
    assert np.allclose(out_full[:100], out_subset)

Writing test_features.py


In [13]:
!pytest test_features.py -v

============================= test session starts =============================
platform win32 -- Python 3.14.6, pytest-9.1.1, pluggy-1.6.0 -- C:\Users\Dell\AppData\Local\Programs\Python\Python314\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\Dell
plugins: anyio-4.14.0
collecting ... collected 5 items

test_features.py::test_ratio_features_shape PASSED                       [ 20%]
test_features.py::test_ratio_features_no_nan PASSED                      [ 40%]
test_features.py::test_gap_features_correct_values PASSED                [ 60%]
test_features.py::test_skew_log_no_negative_output_error PASSED          [ 80%]
test_features.py::test_transformers_stateless_no_leakage PASSED          [100%]

============================== 5 passed in 1.50s ==============================


In [14]:
%%writefile FEATURES.md
# Feature Documentation

## RatioFeatures
Computes ratios between shape-irregularity columns (concavity, concave points) and size columns (area, perimeter), for both mean and worst measurements. Malignant tumors tend to have higher concavity relative to size; the raw columns don't capture this relationship directly. No leakage risk — purely row-wise arithmetic, no fitted statistics.

## WorstMeanGapFeatures
Computes (worst - mean) for each of the 10 base measurements, capturing within-tumor heterogeneity/variability. Malignant cells tend to show more extreme worst-case measurements relative to their mean. No leakage risk — row-wise arithmetic.

## SkewLogTransform
Applies log1p to right-skewed columns (area, concavity) to reduce the influence of outliers and improve linear model performance. No leakage risk — stateless transform, no fitted parameters from training data.

Writing FEATURES.md


In [15]:
print(results)
print("\n5pp+ gain achieved:", results.loc[1, "Gain (pp)"] >= 5)

   Pipeline  Mean ROC-AUC       Std  Gain (pp)
0  Baseline      0.995314  0.005345   0.000000
1  Enriched      0.994927  0.005547  -0.038742

5pp+ gain achieved: False


In [17]:
import joblib
loaded_model = joblib.load("model.joblib")
print(loaded_model)
print(type(loaded_model))

Pipeline(steps=[('features',
                 ColumnTransformer(transformers=[('scaled_raw',
                                                  StandardScaler(),
                                                  ['mean radius',
                                                   'mean texture',
                                                   'mean perimeter',
                                                   'mean area',
                                                   'mean smoothness',
                                                   'mean compactness',
                                                   'mean concavity',
                                                   'mean concave points',
                                                   'mean symmetry',
                                                   'mean fractal dimension',
                                                   'radius error',
                                                   'texture error',
         